In [4]:

import geopandas as gpd 
import pandas as pd
import fiona

RAW_DIR = 'data/raw/'
REF_DIR = 'data/processed/ref/'
GEOM_DIR = 'data/processed/geom/'


# wget https://data.geopf.fr/telechargement/download/ADMIN-EXPRESS-COG/ADMIN-EXPRESS-COG_4-0__GPKG_LAMB93_FXX_2025-01-01/ADMIN-EXPRESS-COG_4-0__GPKG_LAMB93_FXX_2025-01-01.7z
# Documentation : https://geoservices.ign.fr/sites/default/files/2025-06/DL_ADMIN_EXPRESS_4-0.pdf
# File path: ../data/raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg
file_name = "ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg"

file_path = '../' + RAW_DIR + 'territoires/' + file_name
print(f"File path: {file_path}")

%ls ../data/raw/territoires/"ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg"

File path: ../data/raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg
../data/raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg*


In [2]:
ls -lh ../data/processed/geom/

total 1099536
-rw-r--r--  1 jean-jacques  staff   417M Mar  4 15:43 app_communes_geom_original.parquet
-rw-r--r--  1 jean-jacques  staff    24M Mar  4 15:43 app_communes_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   1.4M Mar  4 15:43 collectivites_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   934K Feb 25 19:10 collectivites_topo.json
-rw-r--r--  1 jean-jacques  staff    24M Mar  4 15:43 communes_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff    28M Feb 25 18:51 communes_topo.json
-rw-r--r--  1 jean-jacques  staff   1.6M Mar  4 15:43 departements_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   935K Feb 25 18:59 departements_topo.json
-rw-r--r--  1 jean-jacques  staff   5.2M Mar  4 15:43 epci_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   3.1M Feb 25 19:09 epci_topo.json
-rw-r--r--  1 jean-jacques  staff   619K Mar  4 15:43 regions_geom_simplified.parquet
-rw-r--r--  1 jean-jacques  staff   457K Feb 25 18:59 regions_topo.jso

In [3]:
# Fonction pour ajouter le point representatif (lon, lat)
def add_coord(df):
    df = df.to_crs(4326)
    pts = df.geometry.representative_point()
    df["lon"] = pts.x
    df["lat"] = pts.y
    return df

In [5]:
layers = fiona.listlayers(file_path)

print("Layers trouvés :")
for layer in layers:
    print("-", layer)


Layers trouvés :
- canton
- arrondissement
- arrondissement_municipal
- chef_lieu_d_arrondissement
- chef_lieu_d_arrondissement_municipal
- chef_lieu_de_canton
- chef_lieu_de_collectivite_territoriale
- chef_lieu_de_commune
- chef_lieu_de_commune_associee_ou_deleguee
- chef_lieu_de_departement
- chef_lieu_d_epci
- chef_lieu_de_region
- collectivite_territoriale
- commune
- commune_associee_ou_deleguee
- departement
- epci
- region
- info_metadonnees
- layer_styles


In [12]:
# Chargement des epci et regions 
epci = gpd.read_file(file_path, layer="epci") 
regions = gpd.read_file(file_path, layer="region") 

In [ ]:
# Fonction de Simplification de la précision de geometry

def simplify_geom(gdf, tolerance=20):
    # Reprojection en Lambert 93 car pandas peut sans qu'on le sache le charger en WGS84 
    # la simplification est plus efficace dans un système métrique
    gdf = gdf.to_crs(2154)

    # Correction + simplification + correction
    gdf["geometry"] = (
        gdf["geometry"]
        .buffer(0)
        .simplify(tolerance, preserve_topology=False)
        .buffer(0)
    )

    # Reprojection finale en WGS84 pour le web
    return gdf.to_crs(4326)

In [14]:
# Pré calculer les points représentatifs
epci = add_coord(epci)


In [ ]:
# Vérifier si lon et lat ont bien été ajoutés
epci.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,nature,codes_insee_des_communes_membres,codes_insee_des_departements_membres,code_siren,geometry,lon,lat
0,EPCI____0000000200000172,CC Faucigny-Glières,CC FAUCIGNY-GLIERES,Communauté de communes,74024/74042/74049/74087/74164/74212/74312,74,200000172,"MULTIPOLYGON (((6.45511 46.05427, 6.45502 46.0...",6.428007,46.039487
1,EPCI____0000000200000438,CC du Pays de Pontchâteau Saint-Gildas-des-Bois,CC DU PAYS DE PONTCHATEAU SAINT-GILDAS-DES-BOIS,Communauté de communes,44050/44053/44068/44098/44129/44152/44161/4418...,44,200000438,"MULTIPOLYGON (((-1.98138 47.472, -1.98129 47.4...",-2.111359,47.475060
2,EPCI____0000000200000545,CC des Portes de Romilly-sur-Seine,CC DES PORTES DE ROMILLY-SUR-SEINE,Communauté de communes,10114/10164/10220/10280/10323/10341,10,200000545,"MULTIPOLYGON (((3.70283 48.47208, 3.70041 48.4...",3.714854,48.497358
3,EPCI____0000000200000628,CC Rhône Lez Provence,CC RHONE LEZ PROVENCE,Communauté de communes,84019/84063/84064/84078/84083,84,200000628,"MULTIPOLYGON (((4.77878 44.2107, 4.77819 44.21...",4.738389,44.246116
4,EPCI____0000000200000800,CC Cœur de Sologne,CC COEUR DE SOLOGNE,Communauté de communes,41036/41046/41106/41161/41251/41296,41,200000800,"MULTIPOLYGON (((2.08755 47.58796, 2.08697 47.5...",2.023303,47.573209


In [19]:
regions = add_coord(regions)
# Vérifier si lon et lat ont bien été ajoutés
regions.head()

,cleabs,nom_officiel,nom_officiel_en_majuscules,code_insee,code_siren,geometry,lon,lat
0,REGION__0000000000000024,Centre-Val de Loire,CENTRE-VAL DE LOIRE,24,234500023,"MULTIPOLYGON (((1.62329 48.74003, 1.62291 48.7...",1.716113,47.643887
1,REGION__0000000000000053,Bretagne,BRETAGNE,53,233500016,"MULTIPOLYGON (((-5.00711 48.41601, -5.0071 48....",-2.668907,48.155696
2,REGION__0000000000000044,Grand Est,GRAND EST,44,200052264,"MULTIPOLYGON (((3.64051 48.18462, 3.64011 48.1...",5.759323,48.794784
3,REGION__0000000000000075,Nouvelle-Aquitaine,NOUVELLE-AQUITAINE,75,200053759,"MULTIPOLYGON (((-1.74944 43.38256, -1.74943 43...",0.104648,44.976923
4,REGION__0000000000000076,Occitanie,OCCITANIE,76,200053791,"MULTIPOLYGON (((3.16399 42.45062, 3.16401 42.4...",2.117500,43.689861


### Construire les fichiers avec geometries simplifiées 

In [21]:
gdf_epci = epci[[
    "code_siren",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()


In [22]:
gdf_reg = regions[[
    "code_siren",
    "nom_officiel",
    "geometry",
    "lon",
    "lat"
]].copy()


In [23]:

print(gdf_epci.crs)
print(gdf_reg.crs)


EPSG:4326
EPSG:4326


In [ ]:
# Réduction de la précision pour diminuer la taille des fichiers finaux
# precision 80m pour le contour des epci
epci_s = simplify_geom(gdf_epci, tolerance=80)
# precision 100m pour le contour des regions
reg_s = simplify_geom(gdf_reg, tolerance=100)


In [26]:
print('epci_s:',epci_s.columns)
print('reg_s:',reg_s.columns)

epci_s: Index(['code_siren', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')
reg_s: Index(['code_siren', 'nom_officiel', 'geometry', 'lon', 'lat'], dtype='object')


In [ ]:
geom_path = "../" + GEOM_DIR
# Exporter les données en format parquet 
epci_s.to_parquet(geom_path + "epci_geom_simplified.parquet", index=False)
reg_s.to_parquet(geom_path + "regions_geom_simplified.parquet", index=False)


In [ ]:
ls -lh ../data/processed/geom/